# Abnormal Trading Volume and Future Stock Returns

This notebook contains the complete workflow for AAPL, JPM, and XOM. The formal research period is 2024-2025, with September-December 2023 used only as warm-up data for the 66-day NAV calculation. Saved outputs are retained for review. The licensed CRSP CSV files are not distributed in this public repository.

# 2. Data Pipeline

## Data Acquisition

The raw data was manually downloaded from WRDS CRSP after authentication. The main CSV covers 2024-2025. A second CSV covers September-December 2023 and is used only as warm-up data for the 66-day NAV calculation. The licensed CSV files are not included publicly; `data/README.md` documents the required paths and fields. The fields used are ticker, calendar date, daily price, daily return, and daily trading volume.

In [16]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RAW_FILE = ROOT / "data" / "crsp_simple" / "raw" / "crsp_aapl_jpm_xom_2024_2025.csv"
WARMUP_FILE = ROOT / "data" / "crsp_simple" / "raw" / "crsp_warmup_2023.csv"
OUTPUT_FILE = ROOT / "data" / "crsp_simple" / "processed" / "crsp_aapl_jpm_xom_research.csv"

TICKERS = ["AAPL", "JPM", "XOM"]
WARMUP_START_DATE = "2023-09-01"
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"
NAV_LOOKBACK_DAYS = 66
NAV_PERCENTILE = 0.90
HOLDING_DAYS = 5
ROUND_TRIP_COST = 0.001  # 0.10% per completed trade

## Data Cleaning

The code combines the 2023 warm-up file with the 2024-2025 research file, keeps only the columns needed, gives them shorter names, converts each column to the correct type, removes rows with missing required values, and sorts the observations. The 2023 rows are retained only so that the first 2024 observations have 66 earlier trading days for the NAV calculation. All comparisons and backtest results are restricted to 2024-2025. Weekends and market holidays are not filled because the strategy is defined in trading days. CRSP daily returns are used directly rather than calculated from the raw price column.

In [17]:
def clean_basic_fields(raw_data):
    """Keep the required CRSP columns and convert them to useful data types."""
    data = raw_data[["Ticker", "DlyCalDt", "DlyPrc", "DlyRet", "DlyVol"]].copy()
    data.columns = ["ticker", "date", "price", "ret", "volume"]

    data["date"] = pd.to_datetime(data["date"])
    data["price"] = pd.to_numeric(data["price"], errors="coerce")
    data["ret"] = pd.to_numeric(data["ret"], errors="coerce")
    data["volume"] = pd.to_numeric(data["volume"], errors="coerce")

    data = data[data["ticker"].isin(TICKERS)]
    data = data[data["date"].between(WARMUP_START_DATE, END_DATE)]
    data = data.dropna(subset=["date", "ticker", "price", "ret", "volume"])
    data = data.sort_values(["ticker", "date"]).reset_index(drop=True)
    return data


# Combine the earlier warm-up rows with the main 2024-2025 raw data.
raw = pd.read_csv(RAW_FILE)
warmup_raw = pd.read_csv(WARMUP_FILE)
combined_raw = pd.concat([warmup_raw, raw], ignore_index=True)
data = clean_basic_fields(combined_raw)

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(OUTPUT_FILE, index=False)

print("Rows after cleaning, including warm-up:", len(data))
print("Rows in the 2024-2025 research period:", len(data[data["date"].between(START_DATE, END_DATE)]))
print("Stocks:", list(data["ticker"].unique()))
print("Missing values in required columns:", data.isna().sum().sum())
display(data.head())

Rows after cleaning, including warm-up: 1755
Rows in the 2024-2025 research period: 1506
Stocks: ['AAPL', 'JPM', 'XOM']
Missing values in required columns: 0


,ticker,date,price,ret,volume
0,AAPL,2023-09-01,189.46,0.008463,45678978
1,AAPL,2023-09-05,189.70,0.001267,45209261
2,AAPL,2023-09-06,182.91,-0.035793,81566974
3,AAPL,2023-09-07,177.56,-0.029249,112228896
4,AAPL,2023-09-08,178.18,0.003492,65494876


# 3. Strategy and Backtest

## Strategy

A signal occurs when the stock return is positive and its Normalized Abnormal Volume (NAV) is in the top 10% of the 2024 in-sample NAV distribution. NAV compares today's volume with the mean and standard deviation of the stock's previous 66 trading days. September-December 2023 is used only to provide this history for the beginning of 2024. The 2024 cutoff is kept unchanged for 2025.

The main research comparison controls the time horizon:

- **Signal group:** positive-return days with abnormal volume.
- **Control group:** positive-return days with normal volume.
- **Outcome for both groups:** the same five complete daily returns after buying at the next trading day's close.

Trading action: observe the signal at day *t* close, buy at *t+1* close, earn the returns from *t+2* through *t+6*, and sell at *t+6* close. New signals are ignored while waiting or holding.

In [18]:
def calculate_future_return(returns):
    """Calculate the compounded return over the next five complete trading days."""
    future_return = (
        (1 + returns.shift(-2))
        * (1 + returns.shift(-3))
        * (1 + returns.shift(-4))
        * (1 + returns.shift(-5))
        * (1 + returns.shift(-6))
        - 1
    )
    return future_return


# Calculate Normalized Abnormal Volume (NAV) from the previous 66 trading days.
data["volume_mean_66"] = data.groupby("ticker")["volume"].transform(
    lambda volume: volume.shift(1).rolling(NAV_LOOKBACK_DAYS).mean())
data["volume_std_66"] = data.groupby("ticker")["volume"].transform(
    lambda volume: volume.shift(1).rolling(NAV_LOOKBACK_DAYS).std())
data["nav"] = (data["volume"] - data["volume_mean_66"]) / data["volume_std_66"]

# Use warm-up rows to calculate NAV, but use only 2024 data to choose the cutoff.
analysis_mask = data["date"].between(START_DATE, END_DATE)
nav_cutoff = data.loc[analysis_mask & (data["date"].dt.year == 2024), "nav"].quantile(NAV_PERCENTILE)
data["abnormal_volume"] = data["nav"] >= nav_cutoff
data["up_day"] = data["ret"] > 0
data["signal"] = data["up_day"] & data["abnormal_volume"]
data["future_return_5d"] = data.groupby("ticker")["ret"].transform(calculate_future_return)

# Remove the warm-up rows before the hypothesis comparison and backtest.
research_data = data[analysis_mask].copy()

research_days = research_data[
    research_data["up_day"]
    & research_data["future_return_5d"].notna()
    & research_data["nav"].notna()
].copy()

research_days["year"] = research_days["date"].dt.year
research_days["volume_group"] = np.where(
    research_days["abnormal_volume"],
    "Abnormal volume",
    "Normal volume",)

comparison = research_days.groupby(["year", "volume_group"])["future_return_5d"].agg(
    count="count",
    mean="mean",
    median="median",)

comparison["positive_rate"] = research_days.groupby(
    ["year", "volume_group"]
)["future_return_5d"].apply(lambda returns: (returns > 0).mean())

comparison[["mean", "median", "positive_rate"]] = (
    comparison[["mean", "median", "positive_rate"]] * 100)

print(f"2024 NAV cutoff (90th percentile): {nav_cutoff:.2f}")
comparison.round(2)

2024 NAV cutoff (90th percentile): 0.94


count  mean  median  positive_rate
year volume_group                                       
2024 Abnormal volume     35  0.41    0.51          60.00
     Normal volume      378  0.46    0.46          58.47
2025 Abnormal volume     42  0.91    1.07          64.29
     Normal volume      361  0.31    0.59          58.73

## Equal-Horizon Comparison

This is the main test of the hypothesis. Buy-and-hold is not the main control because it remains invested for a different length of time.

In [19]:
comparison_table = comparison["mean"].unstack()
comparison_table["Abnormal minus normal"] = (
    comparison_table["Abnormal volume"] - comparison_table["Normal volume"])

print("Average future five-day returns (%)")
comparison_table.round(2)

Average future five-day returns (%)


volume_group,Abnormal volume,Normal volume,Abnormal minus normal
year,,,
2024,0.41,0.46,-0.05
2025,0.91,0.31,0.59


## Backtest

Each stock receives one-third of the portfolio each day. An inactive part remains in cash and no leverage is used. A simple 0.10% round-trip trading cost is charged when each trade starts.

In [20]:
def add_strategy_returns(data):
    """Apply the trading rule and calculate each stock's daily strategy return."""

    # Create an empty list to store the results for every stock and year
    all_stocks = []

    # Process each stock and each year separately
    # For example: AAPL 2024, AAPL 2025, JPM 2024, etc.
    for (ticker, year), one_stock in data.groupby(["ticker", "year"]):

        # Make a copy so that the original dataset is not changed directly
        one_stock = one_stock.copy()

        # Start with a strategy return of zero because the strategy is initially in cash
        one_stock["strategy_ret"] = 0.0

        # Mark whether the strategy buys the stock on each date
        one_stock["trade_entry"] = False

        # This becomes True after a signal is observed
        # It tells the strategy to buy at the next trading day's close
        waiting_to_buy = False

        # This records how many holding days remain
        holding_days_left = 0

        # Go through each trading day in date order
        for row in one_stock.index:

            # If a signal was found yesterday, enter the trade today
            if waiting_to_buy:
                one_stock.loc[row, "trade_entry"] = True

                # Deduct the assumed round-trip transaction cost
                # No stock return is earned today because the purchase is made at today's close
                one_stock.loc[row, "strategy_ret"] = -ROUND_TRIP_COST

                # The purchase has now been completed
                waiting_to_buy = False

                # Begin the five-day holding period
                holding_days_left = HOLDING_DAYS

            # If the stock is currently being held, earn today's stock return
            elif holding_days_left > 0:
                one_stock.loc[row, "strategy_ret"] = one_stock.loc[row, "ret"]

                # Reduce the remaining holding period by one day
                holding_days_left = holding_days_left - 1

            # If the strategy is not waiting or holding, check for a new signal
            elif (one_stock.loc[row, "signal"]
                and pd.notna(one_stock.loc[row, "future_return_5d"])):
                # A signal is observed at today's close
                # The strategy will buy at the next trading day's close
                waiting_to_buy = True

        # Add this stock-year result to the results list
        all_stocks.append(one_stock)

    # Combine all stock-year results into one DataFrame
    result = pd.concat(all_stocks)

    # Sort the final dataset and create a clean row index
    result = result.sort_values(["ticker", "date"]).reset_index(drop=True)

    return result


# Extract the calendar year from the date column
# This allows the function to process 2024 and 2025 separately
research_data["year"] = research_data["date"].dt.year

# Apply the trading strategy and save the result in a new DataFrame
strategy_data = add_strategy_returns(research_data)

## Key Results

The table below reports the required cumulative return, annualised Sharpe ratio, and maximum drawdown for the 2024 in-sample period and 2025 validation period.

In [21]:
def performance_statistics(daily_returns):
    """Return cumulative return, annualised Sharpe ratio, and maximum drawdown."""
    cumulative_return = (1 + daily_returns).prod() - 1

    if daily_returns.std() == 0:
        sharpe_ratio = np.nan
    else:
        sharpe_ratio = np.sqrt(252) * daily_returns.mean() / daily_returns.std()

    wealth = (1 + daily_returns).cumprod()
    drawdown = wealth / wealth.cummax() - 1
    max_drawdown = drawdown.min()
    return cumulative_return, sharpe_ratio, max_drawdown


portfolio = strategy_data.groupby(["date", "year"])["strategy_ret"].mean().reset_index()

results = []
for year in [2024, 2025]:
    yearly_returns = portfolio.loc[portfolio["year"] == year, "strategy_ret"]
    cumulative_return, sharpe_ratio, max_drawdown = performance_statistics(yearly_returns)
    number_of_trades = int(
        strategy_data.loc[strategy_data["year"] == year, "trade_entry"].sum()
    )
    results.append({
        "year": year,
        "sample": "In-sample" if year == 2024 else "Validation",
        "trades": number_of_trades,
        "cumulative_return": cumulative_return,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": max_drawdown,
    })

results = pd.DataFrame(results).set_index("year")
results_percent = results.copy()
results_percent[["cumulative_return", "max_drawdown"]] *= 100
display(results_percent.round(2))



,sample,trades,cumulative_return,sharpe_ratio,max_drawdown
year,,,,,
2024,In-sample,25,5.11,1.19,-3.02
2025,Validation,24,7.70,0.97,-6.32


The abnormal-minus-normal difference is the main evidence about the hypothesis. A positive difference supports the idea. The backtest then shows whether the same rule can be converted into a usable trading action after a simple trading-cost assumption.